# OSM BaseData — raw inputs only, no model output

*Notebook style follows Charlotte Ellison's existing RideScore DC notebooks.
The OSM/DC join methodology was developed jointly by Charlotte and EChO Ory.
This notebook was drafted with Claude (Anthropic), editing and structure by
EChO.*

**Just want the data, not the analysis?** Two small extracts are committed in
this folder: `raw_osm_extract.geojson` (OSM street segments) and
`raw_dc_extract.geojson` (DDOT SubBlocks). They are *unjoined* -- if you only
need OSM, use the first file directly; if you want DC attributes attached to
each OSM segment, run this notebook top to bottom (fast, no network) and use
`segments`. Keep reading if you want to check the data's quality before
building a model on it, or if you need a *fresher* pull than the committed
extract.

**What this is:** raw street segment attributes, from OpenStreetMap and
(optionally) DC Open Data, with **no model output of any kind** — no LTS
score, no BNA stress score, no RideScore. Every model in this project should
be able to start from these same raw columns.

**Why two ways to get data, not one file:** a small extract is committed to
git so this notebook works with no network and no wait. A *full-DC* pull is
never committed — at ~100k segments it would blow past GitHub's file-size
limits — so it only ever exists locally, produced on demand by the fetch cell
below. `SUBSET` picks between the two.

## Requirements

Run from the repo root with [uv](https://docs.astral.sh/uv/):

    uv run --group notebooks jupyter lab

or, in VS Code, run `uv sync --group notebooks` once and pick the `.venv`
kernel. (`.venv` is per machine -- rebuild it rather than copying it.)

If running this notebook outside the repo, install manually:

    pip install jupyterlab ipykernel osmnx folium mapclassify matplotlib seaborn numpy python-dateutil geopandas pandas shapely requests


In [ ]:
#import packages
from pathlib import Path #object oriented way to handle file system paths
import geopandas as gpd #geospatial data library
import pandas as pd #data manipulation
import numpy as np #library to work with arrays
import requests #http client for python
import matplotlib.pyplot as plt #data visualization library
import seaborn as sns #high-level data visualization library
import folium #interactive map library acts as Leaflet.js wrapper
from folium import GeoJson #tool for rendering spatial data (points, lines, complex polygons)
from IPython.display import display #library for rich frontend displays
import osmnx as ox #downloads and models street networks from OpenStreetMap

## Parameters

The only settings most people need to touch:

- `SUBSET` -- `1` for the small committed extract (fast, no network),
  `0` for a live pull of all of DC (slow).
- `SUBSET_BBOX` -- the area the small extract covers.
- `INCLUDE_DC_COMPARISON` -- drop the `dc_*` columns if you only want OSM.

Paths and join tuning values are defined here too, so nothing downstream has a
hidden number in it.

In [ ]:
SUBSET = 1
# 1 = small committed extracts (raw_osm_extract.geojson + raw_dc_extract.geojson),
#     joined live by run_join() below -- fast, no network
# 0 = full DC, pulled live from OSM and DC Open Data -- slow; the full-DC join
#     is untested end-to-end (see open questions at the end)

SUBSET_BBOX = (-77.058, 38.9097148, -76.995, 38.9287761)  # WGS84 -- widened to include Rock Creek Trail (west) and the Metropolitan Branch Trail (east)
INCLUDE_DC_COMPARISON = True  # False = OSM columns only, skip the DC join

DATA_DIR = "."
CACHE_DIR = Path("cache")  # notebook-local cache location defined once; gitignored
FULL_DC_CACHE = CACHE_DIR / "osm_full_dc.geojson"      # full-DC OSM pull
FULL_DDOT_CACHE = CACHE_DIR / "ddot_full_dc.geojson"   # full-DC DC Open Data pull

# DDOT centerline layers -- Block (streets only, unbroken at alleys; used by the
# earlier extract and by DC crash data) and SubBlock (finest-grained; splits at
# alleys; includes trails). The join uses SubBlock, with two Block-only
# columns brought in -- see methodology decisions, item 7.
DC_BLOCK_LAYER_URL = (
    "https://maps2.dcgis.dc.gov/dcgis/rest/services/"
    "DCGIS_DATA/Transportation_WebMercator/MapServer/163/query"
)
DC_SUBBLOCK_LAYER_URL = (
    "https://maps2.dcgis.dc.gov/dcgis/rest/services/"
    "DCGIS_DATA/Transportation_WebMercator/MapServer/162/query"
)

# SubBlock road types kept for the join (DDOT codes). Alleys (4), driveways (5)
# and walkways (7) are dropped -- OSM has no equivalent segments, so they could
# only produce false matches.
DC_SUBBLOCK_ROADTYPES = ["1", "2", "3", "6"]  # Street, Service Road, Ramp, Trail

# Block-only attributes worth keeping, pulled in via dc_blockkey
DC_BLOCK_EXTRA_COLUMNS = ["dc_VERTICAL_DEFLECTION", "dc_SURFACE_TYPE"]

# Join tuning -- first-cut values, see open questions before trusting at full-DC scale
JOIN_BUFFER_METERS = 10 # matches dataComparison_v2, which built the earlier extract
JOIN_BEARING_TOLERANCE_DEG = 25

# Which OSM street types count as "the network". A methodology choice -- see
# open questions. Excluded for now: footway, steps, path, service (alleys,
# driveways, parking aisles), pedestrian, motorway.
OSM_HIGHWAY_TYPES = [
    "trunk", "trunk_link", "primary", "primary_link",
    "secondary", "secondary_link", "tertiary", "tertiary_link",
    "residential", "unclassified", "living_street", "cycleway",
]

# Bike trails -- included per team decision. Many trails aren't tagged
# highway=cycleway, so also keep paths/footways where OSM says bikes belong.
OSM_TRAIL_FILTERS = [
    '["highway"="path"]["bicycle"~"^(designated|yes|permissive)$"]',
    '["highway"="footway"]["bicycle"="designated"]',
]

# Alleys: fetched only so streets split where an alley meets them (matching
# DDOT SubBlock segmentation), then dropped -- not part of the network itself.
OSM_SPLIT_ONLY_FILTERS = ['["highway"="service"]["service"="alley"]']

# Committed extract files (SUBSET=1) -- raw and unjoined; run_join() joins them live
RAW_OSM_EXTRACT = Path(DATA_DIR) / "raw_osm_extract.geojson"
RAW_DC_EXTRACT = Path(DATA_DIR) / "raw_dc_extract.geojson"

# Set True only to regenerate the committed extract files from live data
REFRESH_EXTRACT = False

## Getting the data

`SUBSET=1` reads two small committed files -- `raw_osm_extract.geojson` and
`raw_dc_extract.geojson`, *unjoined* -- and joins them live with `run_join()`
below.

`SUBSET=0` fetches both sources live -- OSM via `osmnx`, DC via DDOT's ArcGIS
REST service -- caches them under `cache/`, and runs the same join. The OSM
fetch is confirmed on a single neighborhood, but **a full-city fetch + join
has not been timed or tested end-to-end** -- budget real time for it.

In [ ]:
RAW_OSM_TAGS = [
    "highway", "name", "access", "bicycle", "oneway", "oneway:bicycle", "maxspeed",
    "lanes", "lanes:forward", "lanes:backward",
    "cycleway", "cycleway:left", "cycleway:left:buffer", "cycleway:left:oneway", "cycleway:left:width",
    "cycleway:right", "cycleway:right:buffer", "cycleway:right:oneway", "cycleway:right:width",
    "cycleway:both", "cycleway:both:buffer", "cycleway:both:width", "cycleway:buffer", "cycleway:width",
    "parking:left", "parking:right", "parking:both", "parking:lane:left", "parking:lane:right", "parking:lane:both",
    "parking:left:restriction", "parking:right:restriction", "parking:both:restriction",
    "tracktype", "turn:lanes", "turn:lanes:forward", "turn:lanes:backward", "width", "footway",
]

# osmnx caches raw HTTP responses too; keep them beside -- not mixed into --
# the derived geojson, and pinned rather than left to an implicit "./cache".
CACHE_DIR.mkdir(parents=True, exist_ok=True)
ox.settings.cache_folder = str(CACHE_DIR / "osmnx-http")
ox.settings.useful_tags_way = list(set(ox.settings.useful_tags_way) | set(RAW_OSM_TAGS))

def fetch_osm(bbox=None, place="Washington, District of Columbia, USA"):
    """Pull the street network from OSM, filtered to OSM_HIGHWAY_TYPES plus bike trails.

    bbox: (minx, miny, maxx, maxy) in WGS84 -- e.g. SUBSET_BBOX. If omitted,
    pulls the whole `place` instead (slow; full-DC run is untimed).

    Returns one row per street segment. Segments run between intersections
    (alleys included), split wherever a kept tag changes, one row regardless of
    direction. Columns: osm_id, osm_u, osm_v, osm_<tag> with ':' -> '_',
    geometry. Where a segment merges several OSM ways, osm_id is the first
    way's id. Segments never merge across a change in a kept tag, so any ';'
    in a tag value is OSM's own multi-value syntax (see methodology decisions,
    item 6).
    """
    highway_filter = f'["highway"~"^({"|".join(OSM_HIGHWAY_TYPES)})$"]'
    options = dict(
        custom_filter=[highway_filter] + OSM_TRAIL_FILTERS + OSM_SPLIT_ONLY_FILTERS,
        simplify=False, retain_all=True,
    )
    if bbox is not None:
        graph = ox.graph_from_bbox(bbox, **options)
    else:
        graph = ox.graph_from_place(place, **options)

    # Merge down to intersection-to-intersection segments, but never across a
    # change in any tag we keep -- e.g. a bike lane that starts mid-block splits
    # the segment there instead of blurring into "lane;no".
    tags_present = [t for t in RAW_OSM_TAGS if t in {k for *_, d in graph.edges(data=True) for k in d}]
    graph = ox.simplify_graph(graph, edge_attrs_differ=tags_present)

    # Alleys have done their job (splitting streets where they meet) -- drop them.
    alleys = [(u, v, k) for u, v, k, d in graph.edges(keys=True, data=True)
              if d.get("highway") == "service" and d.get("service") == "alley"]
    graph.remove_edges_from(alleys)

    graph = ox.convert.to_undirected(graph)  # drop the reverse-direction duplicates

    _, edges = ox.graph_to_gdfs(graph)
    edges = edges.reset_index()
    tags = [t for t in RAW_OSM_TAGS if t in edges.columns]
    edges = edges[["osmid", "u", "v", "key"] + tags + ["geometry"]].to_crs("EPSG:4326")

    edges["osmid"] = edges["osmid"].apply(lambda v: v[0] if isinstance(v, list) else v)
    for t in tags:
        edges[t] = edges[t].apply(
            lambda v: ";".join(dict.fromkeys(map(str, v))) if isinstance(v, list) else v
        )

    rename = {"osmid": "osm_id", "u": "osm_u", "v": "osm_v", "key": "osm_key"}
    rename |= {t: "osm_" + t.replace(":", "_") for t in RAW_OSM_TAGS}
    return edges.rename(columns=rename)


In [ ]:
def fetch_dc(bbox=None, layer_url=None, page_size=2000):
    """Pull DDOT centerline segments from DC Open Data's ArcGIS REST service.

    bbox: (minx, miny, maxx, maxy) in WGS84 -- e.g. SUBSET_BBOX. If omitted,
    pulls all of DC.
    layer_url: DC_BLOCK_LAYER_URL (default) or DC_SUBBLOCK_LAYER_URL.
    Returns one row per DDOT segment, WGS84, columns prefixed dc_
    (BLOCKKEY -> dc_blockkey, SUBBLOCKKEY -> dc_subblockkey).
    """
    layer_url = layer_url or DC_BLOCK_LAYER_URL
    params = {
        "where": "1=1", "outFields": "*", "outSR": 4326, "f": "geojson",
        "orderByFields": "OBJECTID",  # stable order so paging doesn't skip or repeat
        "resultRecordCount": page_size,
    }
    if bbox is not None:
        params |= {
            "geometry": ",".join(map(str, bbox)), "geometryType": "esriGeometryEnvelope",
            "inSR": 4326, "spatialRel": "esriSpatialRelIntersects",
        }

    features, offset = [], 0
    while True:
        resp = requests.get(layer_url, params=params | {"resultOffset": offset}, timeout=120)
        resp.raise_for_status()
        page = resp.json()
        if "error" in page:  # ArcGIS reports errors inside a 200 response
            raise RuntimeError(f"DC Open Data query failed: {page['error']}")
        batch = page.get("features", [])
        features += batch
        if len(batch) < page_size:
            break
        offset += page_size

    segs = gpd.GeoDataFrame.from_features(features, crs="EPSG:4326")
    keys = {"BLOCKKEY": "dc_blockkey", "SUBBLOCKKEY": "dc_subblockkey"}
    rename = {c: keys.get(c, f"dc_{c}") for c in segs.columns if c != "geometry"}
    return segs.rename(columns=rename)

## Joining OSM to DC

Each OSM segment is matched to at most one DDOT SubBlock. Candidates are
SubBlocks whose `JOIN_BUFFER_METERS` buffer the segment touches, running
within `JOIN_BEARING_TOLERANCE_DEG` of the same direction (which drops cross
streets at intersections). Two scores come out:

- `match_confidence` -- the share of the OSM segment's length that runs along
  the matched **DC street** (all candidate SubBlocks with the same `ROUTEID`).
  This is "did we find the right street."
- `dc_primary_share` -- the share along the **one SubBlock** whose attributes
  were copied. Below 1 means the OSM segment spans a SubBlock boundary.

The best candidate is the right street first, then the biggest piece of it.
Unmatched OSM segments keep empty `dc_*` columns -- nothing is dropped.


In [ ]:
def segment_bearing(geom):
    """Undirected direction of a line, 0-180 degrees, from its two endpoints."""
    (x1, y1), (x2, y2) = geom.coords[0][:2], geom.coords[-1][:2]
    return np.degrees(np.arctan2(y2 - y1, x2 - x1)) % 180


def run_join(osm, dc, dc_block=None):
    """Attach DDOT SubBlock attributes to each OSM segment (see markdown above).

    osm: output of fetch_osm(). dc: output of fetch_dc(layer_url=DC_SUBBLOCK_LAYER_URL).
    dc_block (optional): output of fetch_dc() for Blocks, for DC_BLOCK_EXTRA_COLUMNS.
    Returns osm with dc_* columns and match_confidence added, same row count, WGS84.
    """
    crs = "EPSG:26985"  # Maryland State Plane, meters -- DC GIS's own projection

    osm_m = osm.to_crs(crs).reset_index(drop=True)
    osm_m["_osm_row"] = osm_m.index
    osm_m["_bearing"] = osm_m.geometry.apply(segment_bearing)

    dc_m = dc[dc["dc_ROADTYPE"].isin(DC_SUBBLOCK_ROADTYPES)].to_crs(crs)
    dc_m = dc_m.explode(index_parts=False).reset_index(drop=True)  # multi-part lines -> single lines
    dc_m["_bearing"] = dc_m.geometry.apply(segment_bearing)

    buffers = gpd.GeoDataFrame(
        {"_bearing": dc_m["_bearing"]},
        geometry=dc_m.geometry.buffer(JOIN_BUFFER_METERS, cap_style="flat"), crs=crs,
    )

    # 1-2: nearby SubBlocks, running the same direction
    cand = gpd.sjoin(osm_m[["_osm_row", "_bearing", "geometry"]], buffers,
                     how="inner", predicate="intersects")
    diff = (cand["_bearing_left"] - cand["_bearing_right"]).abs() % 180
    cand["bearing_diff"] = np.minimum(diff, 180 - diff)  # 170 deg apart = 10 deg apart
    cand = cand[cand["bearing_diff"] <= JOIN_BEARING_TOLERANCE_DEG].copy()

    # 3: how much of each OSM segment lies inside each candidate buffer
    osm_geom = gpd.GeoSeries(osm_m.geometry.loc[cand["_osm_row"]].values, crs=crs)
    buf_geom = gpd.GeoSeries(buffers.geometry.loc[cand["index_right"]].values, crs=crs)
    osm_len = osm_geom.length.values
    safe_len = np.where(osm_len > 0, osm_len, 1)
    inside = osm_geom.intersection(buf_geom).length.values

    # share along this one SubBlock (whose attributes would be copied)
    cand["dc_primary_share"] = np.where(osm_len > 0, inside / safe_len, 0)
    # share along the whole DC street -- all candidate SubBlocks with the same ROUTEID
    cand["_route"] = dc_m["dc_ROUTEID"].loc[cand["index_right"]].values
    cand["_inside"] = inside
    route_inside = cand.groupby(["_osm_row", "_route"], dropna=False)["_inside"].transform("sum")
    cand["match_confidence"] = np.clip(route_inside.values / safe_len, 0, 1)

    # 4: best SubBlock per OSM segment -- right street first, then the biggest piece of it
    best = (cand.sort_values(["match_confidence", "dc_primary_share", "bearing_diff"],
                             ascending=[False, False, True])
                .drop_duplicates("_osm_row")
                [["_osm_row", "index_right", "match_confidence", "dc_primary_share"]])
   
    # 5: copy SubBlock attributes, then Block-only extras via dc_blockkey
    best = best.merge(dc_m.drop(columns=["geometry", "_bearing"]),
                      left_on="index_right", right_index=True).drop(columns="index_right")
    out = osm_m.merge(best, on="_osm_row", how="left")

    if dc_block is not None:
        extra = [c for c in DC_BLOCK_EXTRA_COLUMNS if c in dc_block.columns]
        out = out.merge(dc_block[["dc_blockkey"] + extra].drop_duplicates("dc_blockkey"),
                        on="dc_blockkey", how="left")

    return out.drop(columns=["_osm_row", "_bearing"]).to_crs("EPSG:4326")

In [ ]:
def with_block_extras(dc_sub, dc_block):
    """Add the Block-only columns in DC_BLOCK_EXTRA_COLUMNS to SubBlocks, via dc_blockkey."""
    extra = [c for c in DC_BLOCK_EXTRA_COLUMNS if c in dc_block.columns]
    return dc_sub.merge(dc_block[["dc_blockkey"] + extra].drop_duplicates("dc_blockkey"),
                        on="dc_blockkey", how="left")


if REFRESH_EXTRACT:
    osm_extract = fetch_osm(bbox=SUBSET_BBOX)
    dc_extract = with_block_extras(
        fetch_dc(bbox=SUBSET_BBOX, layer_url=DC_SUBBLOCK_LAYER_URL),
        fetch_dc(bbox=SUBSET_BBOX, layer_url=DC_BLOCK_LAYER_URL),
    )
    osm_extract.to_file(RAW_OSM_EXTRACT, driver="GeoJSON")
    dc_extract.to_file(RAW_DC_EXTRACT, driver="GeoJSON")
    print(f"wrote {len(osm_extract)} OSM segments -> {RAW_OSM_EXTRACT}")
    print(f"wrote {len(dc_extract)} DC SubBlocks -> {RAW_DC_EXTRACT}")
else:
    print("REFRESH_EXTRACT is False -- using the committed extract files as-is.")

## Loading and joining

`SUBSET=1` reads the two committed files; `SUBSET=0` fetches all of DC (cached
under `cache/` after the first run -- delete those files to force a fresh
pull). Either way, `run_join()` then attaches the DC attributes, producing
`segments`, which everything below uses.

In [ ]:
if SUBSET:
    osm_raw = gpd.read_file(RAW_OSM_EXTRACT)
    dc_raw = gpd.read_file(RAW_DC_EXTRACT)
else:
    if FULL_DC_CACHE.exists():
        osm_raw = gpd.read_file(FULL_DC_CACHE)
    else:
        osm_raw = fetch_osm()
        osm_raw.to_file(FULL_DC_CACHE, driver="GeoJSON")
    if FULL_DDOT_CACHE.exists():
        dc_raw = gpd.read_file(FULL_DDOT_CACHE)
    else:
        dc_raw = with_block_extras(fetch_dc(layer_url=DC_SUBBLOCK_LAYER_URL),
                                   fetch_dc(layer_url=DC_BLOCK_LAYER_URL))
        dc_raw.to_file(FULL_DDOT_CACHE, driver="GeoJSON")

segments = run_join(osm_raw, dc_raw) if INCLUDE_DC_COMPARISON else osm_raw
print(f"{len(segments)} segments  (SUBSET={SUBSET}, DC join={'on' if INCLUDE_DC_COMPARISON else 'off'})")

## Checks

Run after loading. The first two are hard checks -- the notebook stops if the
join ever adds/drops rows or segment IDs stop being unique. The rest report
match quality so reviewers can see what the join did.

In [ ]:
id_cols = ["osm_u", "osm_v", "osm_key"]
assert len(segments) == len(osm_raw), "the join added or dropped rows"
dupes = segments.duplicated(subset=id_cols).sum()
assert dupes == 0, f"{dupes} segments share an ID (osm_u, osm_v, osm_key)"
print(f"{len(segments)} segments -- IDs unique, row count preserved by the join\n")

if "dc_subblockkey" in segments.columns:
    has = segments["dc_subblockkey"].notna()
    good = has & (segments["match_confidence"] >= 0.8)
    weak = has & (segments["match_confidence"] < 0.8)
    for label, mask in [("good (>= 0.8)", good), ("weak (< 0.8)", weak), ("no DC match", ~has)]:
        print(f"{label:15} {mask.sum():5}  {mask.mean():.1%}")

    print("\nno DC match, by OSM type:")
    print(segments.loc[~has, "osm_highway"].value_counts().head(10).to_string())
    print("\nweak match, by OSM type:")
    print(segments.loc[weak, "osm_highway"].value_counts().head(10).to_string())
    print("\ngood matches spanning a DDOT SubBlock boundary (dc_primary_share < 0.8):",
          (good & (segments["dc_primary_share"] < 0.8)).sum())
    print("OSM cycleways matched to a DC street (separately mapped bike lanes):",
          (segments["osm_highway"].eq("cycleway") & segments["dc_ROADTYPE"].eq("1")).sum())

## Map: where the join worked, and where it didn't

Blue = good match, orange = weak, purple = no DC match; faint gray = DC
SubBlocks for context. Hover a segment for its OSM name/type, scores, and the
DC street it matched. The map is saved to `cache/join_check.html` (gitignored)
and opened in a browser rather than shown inline -- ~3,000 segments embedded
in the notebook makes it heavy to open and to diff in git.

In [ ]:
if "dc_subblockkey" in segments.columns:
    has = segments["dc_subblockkey"].notna()
    good = has & (segments["match_confidence"] >= 0.8)
    weak = has & (segments["match_confidence"] < 0.8)

    # Esri's light gray canvas: keyless (CartoDB basemaps now require an API
    # key) and neutral enough that the colored match overlays stay readable.
    m = folium.Map(
        location=[38.918, -77.025], zoom_start=15,
        tiles="https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/"
              "World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}",
        attr="Tiles &copy; Esri",  # Esri requires an attribution
    )

    # faint DC SubBlock lines underneath, for context
    dc_context = dc_raw[dc_raw["dc_ROADTYPE"].isin(DC_SUBBLOCK_ROADTYPES)][["dc_ROUTENAME", "geometry"]]
    folium.GeoJson(dc_context.to_json(), name="DC SubBlocks",
                   style_function=lambda x: {"color": "#999999", "weight": 1, "opacity": 0.6}).add_to(m)

    show = segments.assign(match_confidence=segments["match_confidence"].round(2),
                           dc_primary_share=segments["dc_primary_share"].round(2))
    fields = ["osm_name", "osm_highway", "match_confidence", "dc_primary_share", "dc_ROUTENAME"]
    for label, mask, color, weight in [("good", good, "#2166ac", 2),
                                       ("weak", weak, "#f46d43", 4),
                                       ("no match", ~has, "#7b3294", 4)]:
        folium.GeoJson(
            show.loc[mask, fields + ["geometry"]].to_json(), name=f"{label} ({mask.sum()})",
            style_function=lambda x, c=color, w=weight: {"color": c, "weight": w, "opacity": 0.85},
            tooltip=folium.GeoJsonTooltip(fields=fields),
        ).add_to(m)

    folium.LayerControl().add_to(m)
    map_file = CACHE_DIR / "join_check.html"
    m.save(str(map_file))
    print("Map saved -- open in a browser:", map_file.resolve())
else:
    print("DC join is off -- nothing to map.")

## Name agreement (matched segments)

A sanity check on the join that doesn't use geometry: after normalizing
abbreviations, do the OSM name and DDOT `ROUTENAME` agree?

In [ ]:
ABBR = {"STREET": "ST", "AVENUE": "AVE", "ROAD": "RD", "PLACE": "PL", "DRIVE": "DR",
        "TERRACE": "TER", "CIRCLE": "CIR", "COURT": "CT", "PARKWAY": "PKWY",
        "BOULEVARD": "BLVD", "LANE": "LN", "NORTHWEST": "NW", "NORTHEAST": "NE",
        "SOUTHWEST": "SW", "SOUTHEAST": "SE"}

def norm_name(s):
    """'4th Street Northwest' and '4TH ST NW' -> '4TH ST NW'."""
    if not isinstance(s, str):
        return None
    return " ".join(ABBR.get(w, w) for w in s.upper().replace(".", "").split())

has = segments["dc_subblockkey"].notna() if "dc_subblockkey" in segments.columns else None
assert has is not None, "DC join is off -- set INCLUDE_DC_COMPARISON = True"
matched = segments[has].copy()
osm_n, dc_n = matched["osm_name"].map(norm_name), matched["dc_ROUTENAME"].map(norm_name)
both = osm_n.notna() & dc_n.notna()
agree = both & (osm_n == dc_n)

print(f"names agree on {agree.sum()} of {both.sum()} matched segments that have both names")
for label, mask in [("good", matched["match_confidence"] >= 0.8), ("weak", matched["match_confidence"] < 0.8)]:
    print(f"  {label}: {(agree & mask).sum()} of {(both & mask).sum()}")

print("\nname disagreements:")
print(matched.loc[both & ~agree, ["osm_name", "dc_ROUTENAME", "osm_highway", "match_confidence"]]
      .drop_duplicates(["osm_name", "dc_ROUTENAME"]).head(20).to_string())

## Trails present

Rock Creek Trail and the Metropolitan Branch Trail should both appear in the
subset (methodology decisions, item 3).

In [ ]:
trail_names = osm_raw["osm_name"].astype(str)
print(osm_raw[trail_names.str.contains("Rock Creek|Metropolitan Branch", regex=True)]
      .groupby("osm_name")["osm_highway"].value_counts())

## Semicolons in tag values

Should be OSM's own multi-value syntax only (methodology decisions, item 6),
never two ways' values blurred together.

In [ ]:
tag_cols = [c for c in osm_raw.columns if c.startswith("osm_") and c not in ("osm_id", "osm_u", "osm_v", "osm_key")]
blurred = osm_raw[tag_cols].apply(lambda s: s.astype(str).str.contains(";", regex=False)).any(axis=1)
print(f"rows: {len(osm_raw)}   segments with ';' in a tag value (OSM multi-value syntax): {blurred.sum()}")
multi = osm_raw.loc[blurred, tag_cols]
for c in tag_cols:
    vals = multi[c].dropna().astype(str)
    vals = vals[vals.str.contains(";", regex=False)]
    if len(vals):
        print(f"{c}: {len(vals)}  e.g. {vals.value_counts().head(4).to_dict()}")

## Methodology decisions (and why)

Numbers below are for `SUBSET_BBOX`, pulled 2026-09-23 -- OSM is edited
constantly, so a re-pull will differ slightly.

1. **Source: live OSM via osmnx.** Per repeated team decision, OSM is the
   source of truth for segments. The earlier committed extract was built on
   BNA's network (`stress.gpkg`), not a direct OSM pull; this notebook now
   fetches OSM itself, so the extract is reproducible from public data.
2. **Street types:** the types in `OSM_HIGHWAY_TYPES` (trunk through
   residential, plus unclassified, living_street, cycleway), **plus bike
   trails** per team decision: paths where `bicycle` is designated/yes/permissive
   and footways where `bicycle=designated`. Ordinary sidewalks, steps, service
   roads, pedestrian ways and motorways are excluded.
3. **Subset area:** `SUBSET_BBOX` was widened so the committed extract includes
   Rock Creek Trail (west) and the Metropolitan Branch Trail (east) -- both
   confirmed present.
4. **Segmentation:** a segment runs between intersections -- **including where an
   alley meets the street** -- and also splits wherever any kept OSM tag changes
   (e.g. a bike lane starting mid-block), so no segment blurs two different sets
   of attributes. Alleys are fetched only to make those cuts, then dropped; they
   are not segments themselves. Result: 3,005 segments, median 45 m, 185.1 km
   total. The alley split added 713 segments with no change in total length
   (185.1 km with and without), confirming it only cuts streets. Splitting at
   alleys also matches DDOT's SubBlock segmentation. The old BNA-based extract
   (median 14 m) additionally split at every sidewalk, crosswalk and driveway
   junction, which added segments without adding attribute information;
   driveways are better captured later as a count per segment.
5. **One row per segment, not per direction.** Segment identity is
   `osm_u` + `osm_v` + `osm_key`: the OSM node IDs at each end, plus osmnx's
   key to tell apart separate roads joining the same two nodes (e.g. the two
   halves of a divided road -- 3 such pairs in the subset). `osm_id` repeats
   across segments of the same way.
6. **Semicolons in tag values are OSM's own syntax** (e.g. `turn:lanes`
   `left|through;right` = a shared through/right lane), not merged values.
7. **DC attributes come from DDOT Roadway SubBlock** -- DDOT's source layer,
   from which Blocks are derived -- filtered to streets, ramps, service roads
   and trails. Two Block-only attributes relevant to safety
   (`VERTICAL_DEFLECTION` = traffic calming, `SURFACE_TYPE`) are brought in
   through `dc_blockkey`. Both layers were checked on 2026-09-23: identical
   survey dates, every street SubBlock maps to a Block, and they agree on
   ≥97% of values per attribute, so neither is staler; differences reflect
   SubBlock's finer detail.


## What's in here

- `osm_u` + `osm_v` + `osm_key` -- segment identity (end nodes + osmnx key).
  `osm_id` is the OSM way and repeats across segments of the same way.
- `osm_*` -- straight from OpenStreetMap tags (`:` in tag names becomes `_`)
- `dc_*` -- straight from DDOT Roadway **SubBlock**, plus two Block-only
  columns (`dc_VERTICAL_DEFLECTION`, `dc_SURFACE_TYPE`); empty where no match
- `dc_subblockkey` / `dc_blockkey` -- DDOT's identities for the matched piece
- `match_confidence` / `dc_primary_share` -- the join's scores (see "Joining
  OSM to DC")

The DC join runs in both `SUBSET` modes, but has only been checked on the
subset -- see open questions.


In [ ]:
for c in segments.columns:
    print(c)

## Completeness check: OSM vs DC, on matched segments

How often each source fills in a comparable attribute, counted only on
segments that have a DC match. This is the concrete test of whether keeping
DC alongside OSM adds information.

In [ ]:
pairs = [
    ("osm_maxspeed", "dc_SPEEDLIMITS_OB"),
    ("osm_lanes", "dc_TOTALTRAVELLANES"),
    ("osm_cycleway_right", "dc_BIKELANE_CONVENTIONAL"),
    ("osm_width", "dc_TOTALCROSSSECTIONWIDTH"),
]

matched = segments[segments["dc_subblockkey"].notna()]
print(f"{len(matched)} matched segments\n")
for osm_col, dc_col in pairs:
    missing = [c for c in (osm_col, dc_col) if c not in matched.columns]
    if missing:
        print(f"MISSING COLUMN(S): {missing} -- check the name against segments.columns")
        continue
    osm_pct = matched[osm_col].notna().mean()
    dc_pct = matched[dc_col].notna().mean()
    print(f"{osm_col:<22} populated {osm_pct:>6.1%}   |   {dc_col:<28} populated {dc_pct:>6.1%}")


## Open questions for the group

1. **Which source wins when OSM and DC disagree**, per attribute?
2. **Weak matches (2.7% of the subset)** are mostly curved roads, trails and
   short stubs at intersections, where a straight endpoint-to-endpoint bearing
   misdescribes the line. Drop, flag, trust as-is, or improve the join?
3. **No DC match (3.9%)** -- what happens to these segments?
4. **Separately mapped bike lanes:** 270 OSM cycleways run alongside a DC
   street and pick up that street's attributes. Treat them as their own
   facility, or fold them into the parent street?
5. **Should `path` and `service` (alleys) be part of the network?** Right now
   alleys only split streets and are then dropped, and plain paths are kept
   only when OSM says bikes belong.
6. **Driveways** -- captured later as a count per segment rather than as
   splits (decisions, item 4). From which source?
7. **Overrides process.** Proposed order when data is wrong: fix it in OSM
   upstream -> report DDOT errors to DDOT -> a local overrides CSV only as a
   last resort. Needs agreement before anyone starts an overrides file.
8. **Naming conflict near McMillan Reservoir:** OSM appears to have two
   parallel ways here, one named 4th St, one 5th St NE, and a bike path. Check
   on the map before deciding whether it's an OSM fix, a DDOT report, or an
   override.
9. **DDOT's traffic volumes (`AADT`) are from 2020**, a pandemic year when DC
   traffic was well below normal. Check DDOT's traffic volume service for newer
   counts before any model leans on AADT. Pavement condition dates to 2023 and
   road roughness to 2019.
10. **The full-DC run is untested and untimed.** `SUBSET=0` runs the same
    fetch + buffer/bearing join, but it has only been checked on the subset.
    Test it with a real time budget before depending on it.
11. **BNA needs more than this file has**: destination data (schools, jobs,
    parks, retail) and census population, for the connectivity side of BNA --
    a separate data source, not a column added here.
